In [42]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_score, KFold, LeaveOneGroupOut
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import clone
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as torchopt
from torch.utils.data import Dataset, DataLoader, Subset


In [ ]:
base_dir = Path("data")
guided_dir = base_dir / "guided"
freemoves_dir = base_dir / "freemoves"


file_path_EMG = guided_dir / "guided_dataset_X.npy"
dataEMG = np.load(file_path_EMG)

file_path_HAND = guided_dir / "guided_dataset_y.npy"
dataHAND = np.load(file_path_HAND)

file_path_TestGuided = freemoves_dir / "guided_testset_X.npy"
GuidedTest = np.load(file_path_TestGuided)

file_path_EMG_free = freemoves_dir / "freemoves_dataset_X.npy"
dataEMGFree = np.load(file_path_EMG_free)

file_path_HAND_free = freemoves_dir / "freemoves_dataset_y.npy"
dataHANDFree = np.load(file_path_HAND_free)

file_path_TestFree = freemoves_dir / "freemoves_testset_X.npy"
FreeTest = np.load(file_path_TestFree)

In [44]:
cutoff_high = 20
cutoff_low = 500
order_high = 2
order_low = 4

b_high, a_high = butter(order_high, cutoff_high / (1024 / 2), btype='highpass')
b_low, a_low = butter(order_low, cutoff_low / (1024 / 2), btype='lowpass')

def filter_emg(data):
    high = filtfilt(b_high, a_high, data)
    return filtfilt(b_low, a_low, high)

dataEMG = filter_emg(dataEMG)
dataEMGFree = filter_emg(dataEMGFree)

print("Missing guided EMG:", np.isnan(dataEMG).sum())
print("Missing free EMG:", np.isnan(dataEMGFree).sum())

n_sessions, n_electrodes, n_samples = dataEMG.shape



Missing guided EMG: 0
Missing free EMG: 0


In [45]:
ws = 500
step = 250
n_samples = 230000

def window_data(emg, hand):
    n_sessions, n_electrodes, n_samples = emg.shape
    n_joints = dataHAND.shape[1]  

    emg_windows = {}
    hand_windows = {}

    for s in range(n_sessions):
        emg_windows[s] = {}
        hand_windows[s] = {}
        for e in range(n_electrodes):
            emg_windows[s][e] = [
                emg[s, e, start:start+ws]
                for start in range(0, n_samples - ws + 1, step)
            ]
        for j in range(n_joints):
            hand_windows[s][j] = [
                hand[s, j, start:start+ws]
                for start in range(0, n_samples - ws + 1, step)
            ]
    return emg_windows, hand_windows

emg_windows, hand_windows = window_data(dataEMG, dataHAND)
emg_windows_Free, hand_windows_Free = window_data(dataEMGFree, dataHANDFree)

In [46]:
windows_list = []
hand_last = []
groups = []

for s in range(5):
    for w in range(918):
        groups.append(s)
        emg_stack = np.stack([emg_windows[s][e][w] for e in range(n_electrodes)], axis=0)
        windows_list.append(emg_stack)
        last_samples = [hand_windows[s][j][w][-1] for j in range(51)]
        hand_last.append(last_samples)

X = np.stack(windows_list, axis=0)
Y = np.array(hand_last)
groups = np.array(groups)

print("Guided — X.shape:", X.shape, "Y.shape:", Y.shape, "groups.shape:", groups.shape)



Guided — X.shape: (4590, 8, 500) Y.shape: (4590, 51) groups.shape: (4590,)


In [47]:
windows_list_free = []
hand_last_free = []
groups_free = []

for s in range(5):
    for w in range(918):
        groups_free.append(s)
        emg_stack = np.stack([emg_windows_Free[s][e][w] for e in range(n_electrodes)], axis=0)
        windows_list_free.append(emg_stack)
        last_samples = [hand_windows_Free[s][j][w][-1] for j in range(51)]
        hand_last_free.append(last_samples)

X_free = np.stack(windows_list_free, axis=0)
Y_free = np.array(hand_last_free)
groups_free = np.array(groups_free)

print("Free — X.shape:", X_free.shape, "Y.shape:", Y_free.shape, "groups.shape:", groups_free.shape)

Free — X.shape: (4590, 8, 500) Y.shape: (4590, 51) groups.shape: (4590,)


In [48]:
logo = LeaveOneGroupOut()
logo.get_n_splits(X, Y, groups)

print("Guided:")
for i, (train_index, test_index) in enumerate(logo.split(X, Y, groups)):
    print(f"Fold {i}: Train groups={np.unique(groups[train_index])}, Test group={np.unique(groups[test_index])}")

print("\nFree:")
for i, (train_index, test_index) in enumerate(logo.split(X_free, Y_free, groups_free)):
    print(f"Fold {i}: Train groups={np.unique(groups_free[train_index])}, Test group={np.unique(groups_free[test_index])}")


Guided:
Fold 0: Train groups=[1 2 3 4], Test group=[0]
Fold 1: Train groups=[0 2 3 4], Test group=[1]
Fold 2: Train groups=[0 1 3 4], Test group=[2]
Fold 3: Train groups=[0 1 2 4], Test group=[3]
Fold 4: Train groups=[0 1 2 3], Test group=[4]

Free:
Fold 0: Train groups=[1 2 3 4], Test group=[0]
Fold 1: Train groups=[0 2 3 4], Test group=[1]
Fold 2: Train groups=[0 1 3 4], Test group=[2]
Fold 3: Train groups=[0 1 2 4], Test group=[3]
Fold 4: Train groups=[0 1 2 3], Test group=[4]


In [49]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from numpy.fft import rfft, rfftfreq
import pywt 

class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.sigma = 0.05
        self.features_name = [
            'MoyenneValAbsolue', 'RacineMoyenneCarre', 'Variance', 'StandartDeviation', 'NbrChangementSigne', 'ProportionMoyenne',
            'Median', 'Percentile10', 'Percentile90', 'EcartInterquartile',
            'Skewness', 'Kurtosis',
            'Slope', 'Intercept', 'R2',
            'AutoCorr1', 'NbrPics',
            'SpectralCentroid', 'SpectralEntropy',
            'LongueurOnde', 'ZeroCrossing', 'lopeSignChange', 'EMAV', 'EWLs'
            
        ]


    def fit(self, X, y=None):
        return self

    def transform(self, X):
        self.sigma = np.std(X)
        windows, channels, _ = X.shape
        features_number = len(self.features_name)
        transformed = np.zeros((windows, channels * features_number))

        for window in range(windows):
            for channel in range(channels):
                features = self.compute_features(X[window, channel, :])
                transformed[window,
                            channel*features_number:(channel+1)*features_number] = features
        return transformed

        

    def get_feature_names_out(self, input_features=None):
        names = []
        for ch in range(self.n_channels_):
            for fn in self.features_name:
                names.append(f"ch{ch}_{fn}")
        return np.array(names)

    def compute_features(self, x):
        t = np.arange(len(x))
        T = 0.01 * np.max(np.abs(x))

        MoyenneValAbsolue    = np.mean(np.abs(x))
        RacineMoyenneCarre   = np.sqrt(np.mean(x**2))
        Variance             = np.var(x, ddof=1)
        StandartDeviation    = np.std(x, ddof=1)
        NbrChangementSigne   = np.sum(np.diff(np.sign(x)) != 0)
        ProportionMoyenne    = np.sum(np.abs(x) > self.sigma) / len(x)

        Median               = np.median(x)
        Percentile10         = np.percentile(x, 10)
        Percentile90         = np.percentile(x, 90)
        EcartInterquartile   = Percentile90 - Percentile10
        Skewness             = skew(x)
        Kurtosis             = kurtosis(x)

        Slope, Intercept     = np.polyfit(t, x, 1)
        corr                 = np.corrcoef(t, x)[0, 1]
        R2                   = corr**2 if not np.isnan(corr) else 0.0

        AutoCorr1            = np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 1 else 0.0
        peaks, _             = find_peaks(x)
        NbrPics              = len(peaks)

        Xf                   = np.abs(rfft(x))
        freqs                = rfftfreq(len(x), d=1.0)
        SpectralCentroid     = (freqs * Xf).sum() / (Xf.sum() + 1e-12)
        p                    = Xf / (Xf.sum() + 1e-12)
        SpectralEntropy      = -np.sum(p * np.log2(p + 1e-12))
        
        LongueurOnde = np.sum(np.abs(np.diff(x)))
        ZeroCrossing = np.sum(
        ((x[:-1] * x[1:] < 0) & (np.abs(x[:-1] - x[1:]) >= T)).astype(int))
        SlopeSignChange = np.sum(
        (((x[2:] - x[1:-1]) * (x[1:-1] - x[:-2]) < 0) & ((np.abs(x[2:] - x[1:-1]) >= T) | (np.abs(x[1:-1] - x[:-2]) >= T))).astype(int))

        L = len(x)
        weights = np.ones(L) * 0.5
        weights[int(0.2*L):int(0.8*L)] = 0.75
        EMAV = np.mean(weights * np.abs(x))

        coeffs = pywt.wavedec(x, 'bior3.3', level=4)
        EWLs = np.sum([np.sum(c**2) for c in coeffs[1:]])

        
        

        return [
            MoyenneValAbsolue, RacineMoyenneCarre, Variance, StandartDeviation, NbrChangementSigne, ProportionMoyenne,
            Median, Percentile10, Percentile90, EcartInterquartile,
            Skewness, Kurtosis,
            Slope, Intercept, R2,
            AutoCorr1, NbrPics,
            SpectralCentroid, SpectralEntropy,
            LongueurOnde, ZeroCrossing, SlopeSignChange, EMAV, EWLs
            
        ]

In [24]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import Ridge
from joblib import Memory
import joblib

#def f_regression_multioutput(X, Y):
#   fs, ps = zip(*(f_regression(X, Y[:, i]) for i in range(Y.shape[1])))
#  fs = np.mean(fs, axis=0)
# ps = np.mean(ps, axis=0)
#return fs, ps

    
pipe = Pipeline([
    ('feat',   FeatureExtractor()),
    ('pca',    PCA(n_components=0.95)), 
#    ('select', SelectKBest(score_func=f_regression_multioutput)),  
    ('scale',  StandardScaler()),
    ('model',  'passthrough')
])

param_grid = [

  { 
#    'select__k': [7],
    'model': [DecisionTreeRegressor(random_state=42)],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_leaf': [1, 3, 5],
  },

  {  
    
 #   'select__k': [7],
    'model': [RandomForestRegressor(random_state=42, n_jobs=-1)],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [10, 20],
    'model__max_features': ['sqrt', 'log2'],
  },



 {
     'model': [Ridge()],
     'model__alpha': [0.1, 1.0, 10.0]  
  }


]


gs = GridSearchCV(
    pipe,
    param_grid,
    cv=logo,                      
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=3
)



gs.fit(X, Y, groups=groups)


results = pd.DataFrame(gs.cv_results_)
results['RMSE'] = -results['mean_test_score']
results = results.sort_values('RMSE')
print(results[['params', 'RMSE']].to_string(index=False))
joblib.dump(gs, "gridsearch_results.pkl")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
                                                                                                                                         params     RMSE
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 200} 3.861640
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 200} 3.861640
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 100} 3.884214
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 100} 3.884214
 {'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 50} 3.892403
 {'model': RandomFor

['gridsearch_results.pkl']

In [25]:
pipeFree = Pipeline([
    ('feat',   FeatureExtractor()),
    ('pca',    PCA(n_components=0.95)), 
#    ('select', SelectKBest(score_func=f_regression_multioutput)),  
    ('scale',  StandardScaler()),
    ('model',  'passthrough')
])

param_grid = [

  { 
#    'select__k': [7],
    'model': [DecisionTreeRegressor(random_state=42)],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_leaf': [1, 3, 5],
  },

  {  
    
 #   'select__k': [7],
    'model': [RandomForestRegressor(random_state=42, n_jobs=-1)],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [10, 20],
    'model__max_features': ['sqrt', 'log2'],
  },



 {
     'model': [Ridge()],
     'model__alpha': [0.1, 1.0, 10.0]  
  }


]


gs_free = GridSearchCV(
    pipeFree,
    param_grid,
    cv=logo,                      
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=3
)


gs_free.fit(X_free, Y_free, groups=groups_free)


results_free = pd.DataFrame(gs_free.cv_results_)
results_free['RMSE'] = -results_free['mean_test_score']
results_free = results_free.sort_values('RMSE')
print("=== Résultats GridSearch - Libre ===")
print(results_free[['params', 'RMSE']].to_string(index=False))
joblib.dump(gs_free, "gridsearch_results_free.pkl")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
=== Résultats GridSearch - Libre ===
                                                                                                                                         params     RMSE
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 200} 7.118755
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 200} 7.118755
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 100} 7.134938
{'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 100} 7.134938
 {'model': RandomForestRegressor(n_jobs=-1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimator

['gridsearch_results_free.pkl']

In [58]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import LeaveOneGroupOut
from joblib import dump
import os
import torch.nn as nn

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")




class EMGData(Dataset):
    def __init__(self, features, labels):
        self.feats = torch.tensor(features, dtype=torch.float32)
        self.labs = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.feats.shape[0]

    def __getitem__(self, id):
        return self.feats[id], self.labs[id]

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.extract = nn.Sequential(
            nn.Conv1d(8,64,7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64,128,5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128,256,3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(10)
        )
        self.final_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*10,512),
            nn.ReLU(),
            nn.Linear(512,128),
            nn.ReLU(),
            nn.Linear(128,51)
        )

    def forward(self, x):
        rep = self.extract(x)
        out = self.final_layer(rep)
        return out

def execute_training(net, dloader_tr, dloader_vl, E=60):
    loss_fn = nn.MSELoss()
    opt = torchopt.Adam(net.parameters(), lr=1e-3)
    sched = torchopt.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

    net.to(DEV)
    lowest_rmse = 99999.0
    save_best = None
    bad_rounds = 0
    max_wait = 10

    for e in range(E):
        net.train()
        for x_, y_ in dloader_tr:
            x_, y_ = x_.to(DEV), y_.to(DEV)
            opt.zero_grad()
            err = loss_fn(net(x_), y_)
            err.backward()
            opt.step()

        
        net.eval()
        preds_buf, labs_buf = [], []
        with torch.no_grad():
            for u, v in dloader_vl:
                u = u.to(DEV)
                y_hat = net(u).cpu()
                preds_buf.append(y_hat)
                labs_buf.append(v)

        preds_val = torch.cat(preds_buf).numpy()
        labs_val = torch.cat(labs_buf).numpy()
        rmse_now = np.sqrt(mean_squared_error(labs_val, preds_val))
        sched.step(rmse_now)

        if rmse_now < lowest_rmse:
            lowest_rmse = rmse_now
            save_best = net.state_dict()
            bad_rounds = 0
        else:
            bad_rounds += 1
            if bad_rounds >= max_wait:
                print("Early stop:", e)
                break

    if save_best:
        net.load_state_dict(save_best)
    return net, lowest_rmse


        
def predict(
    X, Y, groups, model_class, name_prefix="guided", batch_size=64, epochs=60, verbose=True
):

    scaler_Y = StandardScaler()
    Y_scaled = scaler_Y.fit_transform(Y)

    full_data = EMGData(X, Y_scaled)
    logo = LeaveOneGroupOut()

    y_true_all = []
    y_pred_all = []
    rmse_folds = []
    trained_models = []

    os.makedirs("models", exist_ok=True)

    for i, (train_idx, test_idx) in enumerate(logo.split(X, Y_scaled, groups)):
        if verbose:
            print(f"\n Fold {i+1} — Train groups: {np.unique(groups[train_idx])}, Test: {np.unique(groups[test_idx])}")

        tr_dl = DataLoader(Subset(full_data, train_idx), batch_size=batch_size, shuffle=True)
        vl_dl = DataLoader(Subset(full_data, test_idx), batch_size=batch_size, shuffle=False)

        model = model_class()
        model, rmse = execute_training(model, tr_dl, vl_dl, E=epochs)
        rmse_folds.append(rmse)
        trained_models.append(model)

        
        model_path = f"models/{name_prefix}_fold{i+1}.pt"
        torch.save(model.state_dict(), model_path)
        if verbose:
            print(f" Modèle sauvegardé → {model_path}")

        model.eval()
        with torch.no_grad():
            x_test = torch.tensor(X[test_idx], dtype=torch.float32).to(DEV)
            y_pred_scaled = model(x_test).cpu().numpy()

        y_pred = scaler_Y.inverse_transform(y_pred_scaled)
        y_true = Y[test_idx]

        y_pred_all.append(y_pred)
        y_true_all.append(y_true)

    y_pred_all = np.vstack(y_pred_all)
    y_true_all = np.vstack(y_true_all)

 
    dump(scaler_Y, f"models/{name_prefix}_scaler_Y.joblib")

    if verbose:
        rmse_total = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
        r2_total = r2_score(y_true_all, y_pred_all)
        print(f"\n Moyenne RMSE folds : {np.mean(rmse_folds):.4f}")
        print(f" Global RMSE        : {rmse_total:.4f} | R² : {r2_total:.4f}")

    return y_true_all, y_pred_all, trained_models, scaler_Y


In [ ]:
y_true_guided, y_pred_guided, models_guided, scaler_guided = predict(
    X, Y, groups, Net, name_prefix="guided"
)

y_true_free, y_pred_free, models_free, scaler_free = predict(
    X_free, Y_free, groups_free, Net, name_prefix="free"
)


In [ ]:
def get_best_model_params(results_df, model_type):
    return results_df[results_df['params'].apply(lambda p: isinstance(p['model'], model_type))] \
                     .sort_values('RMSE') \
                     .iloc[0]['params']


best_tree_params_guided = get_best_model_params(results, DecisionTreeRegressor)
best_rf_params_guided  = get_best_model_params(results, RandomForestRegressor)
best_ridge_params_guided    = get_best_model_params(results, Ridge)

best_tree_params_free = get_best_model_params(results_free, DecisionTreeRegressor)
best_rf_params_free  = get_best_model_params(results_free, RandomForestRegressor)
best_ridge_params_free    = get_best_model_params(results_free, Ridge)

def build_model_pipeline(model_params):

    model = model_params['model']
    for key, value in model_params.items():
        if key.startswith('model__'):
            setattr(model, key.split('__')[1], value)

    pipe = make_pipeline(
        Pipeline([
            ('feat', FeatureExtractor()),
            ('pca', PCA(n_components=0.95)),
            ('scale', StandardScaler())
        ]),
        model
    )
    
    return pipe





tree_pipe_guided = build_model_pipeline(best_tree_params_guided).fit(X, Y)
rf_pipe_guided   = build_model_pipeline(best_rf_params_guided).fit(X, Y)
ride_pipe_guided = build_model_pipeline(best_ridge_params_guided).fit(X, Y)

tree_pipe_free   = build_model_pipeline(best_tree_params_free).fit(X_free, Y_free)
rf_pipe_free     = build_model_pipeline(best_rf_params_free).fit(X_free, Y_free)
ride_pipe_free   = build_model_pipeline(best_ridge_params_free).fit(X_free,Y_free)





## Moyenne

In [ ]:
GuidedTest_processed = GuidedTest.reshape(-1, 8, 500)

full_data = EMGData(X, Y)
train_dl = DataLoader(full_data, batch_size=64, shuffle=True)

model_final = Net()
model_final, _ = execute_training(model_final, train_dl, train_dl, E=60)

tree_pred_test = tree_pipe.predict(GuidedTest_processed)
rf_pred_test   = rf_pipe.predict(GuidedTest_processed)

with torch.no_grad():
    x_tensor_test = torch.tensor(GuidedTest_processed, dtype=torch.float32).to(DEV)
    nn_pred_test = model_final(x_tensor_test).cpu().numpy()

pred_test_avg = average_ensemble_prediction(
    [tree_pred_test, rf_pred_test, nn_pred_test],
    y_true=None,
    verbose=False
)



import pandas as pd
pd.DataFrame(pred_test_avg).to_csv("submission_guided_avg.csv", index=False, header=False)

print("Partie guided, 1660 lignes × 51 colonnes générées.")


In [ ]:
FreeTest_processed = FreeTest.reshape(-1, 8, 500)  


full_data_free = EMGData(X_free, Y_free)
train_dl_free = DataLoader(full_data_free, batch_size=64, shuffle=True)

model_free = Net()
model_free, _ = execute_training(model_free, train_dl_free, train_dl_free, E=60)


tree_pred_test_free = tree_pipe_free.predict(FreeTest_processed)
rf_pred_test_free   = rf_pipe_free.predict(FreeTest_processed)

with torch.no_grad():
    x_tensor_free = torch.tensor(FreeTest_processed, dtype=torch.float32).to(DEV)
    nn_pred_test_free = model_free(x_tensor_free).cpu().numpy()


pred_test_avg_free = average_ensemble_prediction(
    [tree_pred_test_free, rf_pred_test_free, nn_pred_test_free],
    y_true=None,
    verbose=False
)


import pandas as pd
pd.DataFrame(pred_test_avg_free).to_csv("submission_free_avg.csv", index=False, header=False)

print(" Partie free, 1540 lignes × 51 colonnes générées")


In [ ]:

guided_df = pd.read_csv("submission_guided_avg.csv", header=None)
free_df   = pd.read_csv("submission_free_avg.csv", header=None)


assert guided_df.shape == (1660, 51), f"Expected (1660, 51), got {guided_df.shape}"
assert free_df.shape == (1540, 51), f"Expected (1540, 51), got {free_df.shape}"


final_predictions = pd.concat([guided_df, free_df], axis=0)


assert final_predictions.shape == (3200, 51), "Erreur taille"


final_predictions.to_csv("team_submission.csv", index=False, header=False)
print("Fichier final généré")

## Meta-learner

In [ ]:
tree_pred_guided = tree_pipe_guided.predict(X)
rf_pred_guided   = rf_pipe_guided.predict(X)
ride_pred_guided = ride_pipe_guided.predict(X)

tree_pred_free   = tree_pipe_free.predict(X_free)
rf_pred_free     = rf_pipe_free.predict(X_free)
ride_pred_free   = ride_pipe_free.predict(X_free)